In [19]:
from collections import OrderedDict
import numpy as np
import qiskit_metal as metal

import math
from shapely.ops import unary_union
from shapely.geometry import box, JOIN_STYLE
from qiskit_metal import draw, Dict
from qiskit_metal import designs
from qiskit_metal.qlibrary.core import QComponent
from qiskit_metal import Dict
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal import draw
from qiskit_metal.qlibrary.core.base import QComponent
from purcell_resonator_candycane import CandyCanePurcell
from airbridge_qcomponent import route_airbridgesQ, single_AirbridgeQ

import importlib
import airbridge_route

importlib.reload(airbridge_route)

from airbridge_route import _fillet_centerline, route_airbridges
from airbridge_route import route_airbridges



In [20]:
design = designs.design_planar.DesignPlanar()
design.overwrite_enabled = True   
design._chips.main.size.size_x = '5mm'
design._chips.main.size.size_y = '5mm' 
gui = metal.MetalGUI(design)

11:18PM 56s INFO [_start_renderers]: Renderer=hfss skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
11:18PM 56s INFO [_start_renderers]: Renderer=q3d skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
11:18PM 56s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).
11:18PM 56s INFO [_start_renderers]: Renderer=aedt_q3d skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').
11:18PM 56s INFO [_start_renderers]: Renderer=aedt_hfss skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').


In [21]:
%metal_print SQUID

In [22]:
class Squid(QComponent):

    default_options = Dict(
        pos_x = "0um",
        pos_y = "0um",

        t_horizontal_width="2um",
        t_horizontal_length="2um",

        t_vertical_width="2um",
        t_vertical_length="6um",

        side_arm_width="10um",
        side_arm_length="52.5um",
        side_arm_move_y="0um",

        stem_x="0um",
        stem_width="25um",
        attach_y="0um",

        added_const="0.5um",

        t_layer="5",
        
        squid_opt = Dict(
            plate1_width="3um",
            plate1_height="19um",

            segment_a_length="4um",
            segment_a_width="3um",

            JJ_gap="2um",

            segment_c_width="4um",
            segment_c_height="20um",

            segment_d_length="7.5um",
            segment_d_width="4um",

            plate2_width="7um",
            plate2_height="12um",

            extension_width="7um", #extension is the extra rectangle that attaches to the lower part of plate 1 that covers the T in the Si layer
            extension_length="10um",
    
    
            sides_lengths=("4um", "2.07571198628um", "2.07571198628um"), #this calls on my function to make the JJ and it takes base, and the two sides as inputs (of a triangle)

            #two rectangles are specified cause there are two different jjs with a different width on either side of squid
            rectangle1_width="0.4um",
            rectangle1_length="1.36um",
    
            rectangle2_width="0.17um",
            # Compensate the narrower finger's later triangle contact point so
            # its Dolan opening matches junction 1 at exactly 140 nm.
            rectangle2_length="1.32805875um",

            rotation=0,
    
            layer_60_length_1="1.5um",
            layer_60_length_2="1.46805875um", #the jjs live on a different layer than the squid.
        ),

        transmon_side = "1",

        layer = "20"
    )
    
    component_metadata = Dict(
        short_name="SQUID",
        _qgeometry_table_poly="True"
    )

    def make(self):
        p = self.parse_options()

        pin_center = self.make_connection_arm(p, p.stem_x, p.attach_y, p.stem_width)
        
        self.extension_plate_center = self.make_squid(p, p.squid_opt, pin_center[0], pin_center[1])

        self.make_t_si(p, self.extension_plate_center[0], self.extension_plate_center[1])


        
    def make_squid(self, p, squid_opt, attach_x, attach_y):
        s = squid_opt
        #PLATE 1:
        #init pos of squid is determined by plate 1
        x_1 = 0
        y_1 = 0
        mirror = (p.transmon_side == -1)

        plate1 = draw.rectangle(s.plate1_width, s.plate1_height, x_1, y_1)

        #SEGMENT A: 
        x_a_upper = x_1 + (s.plate1_width/2) + (s.segment_a_width/2)
        y_a_upper = y_1 + (s.plate1_height/2) - (s.segment_a_length/2)
        segment_a = draw.rectangle(
                    s.segment_a_width,
                    s.segment_a_length,
                    x_a_upper,
                    y_a_upper
                        )

        #SEGMENT C:
        x_c = x_a_upper + (s.segment_a_width/2) + s.JJ_gap + (s.segment_c_width/2)
        y_c = y_a_upper + (s.segment_a_length/2) - (s.segment_c_height/2)
        
        segment_c = draw.rectangle(
            s.segment_c_width,
            s.segment_c_height,
            x_c,
            y_c
        )

        #SEGMENT A LOWER:
        y_a_lower = y_c - (s.segment_c_height/2) + (s.segment_a_length/2)
        segment_a_lower = draw.rectangle(
            s.segment_a_width,
            s.segment_a_length,
            x_a_upper,
            y_a_lower
        )

        #SEGMENT D: 
        x_d = x_c + (s.segment_c_width/2) + (s.segment_d_length/2) -0.001
        segment_d = draw.rectangle(
            s.segment_d_length,
            s.segment_d_width,
            x_d,
            y_c
        )

        #PLATE 2:
        x_plate2 = x_d + (s.segment_d_length/2) + (s.plate2_width/2)
                
        plate2 = draw.rectangle(
            s.plate2_width,
            s.plate2_height,
            x_plate2,
            y_c
        )

        #EXTENSION PLATE:
        x_extension_plate = x_1 + (s.plate1_width/2) - (s.extension_width/2)
        y_extension_plate = y_1 - (s.plate1_height/2) - (s.extension_length/2)
        extension_plate = draw.rectangle(
            s.extension_width,
            s.extension_length,
            x_extension_plate,
            y_extension_plate
        )

        #Make the squid with all of the components, no JJ yet
        squid_no_jj = draw.union(
                    plate1,
                    segment_a,
                    segment_a_lower,
                    extension_plate,
                    segment_c,
                    segment_d,
                    plate2
                )

        #Now we proceed to make the two JJs.
        #JJ1
        jj1 = self.jj_squid(
                    s.sides_lengths,
                    s.rectangle1_width,
                    s.rectangle1_length
                )

        jj1_layer60 = self.jj_squid(
                    s.sides_lengths,
                    s.rectangle1_width,
                    s.layer_60_length_1
                ) #the much longer jj1 that lives in a different layer
        
        jj_1 = draw.rotate(jj1, 270, origin=(0, 0))
        jj_1_layer60 = draw.rotate(jj1_layer60, 270, origin=(0, 0))

        #JJ2
        jj2 = self.jj_squid(
                    s.sides_lengths,
                    s.rectangle2_width,
                    s.rectangle2_length
                )
        jj2_layer60 = self.jj_squid(
                            s.sides_lengths,
                            s.rectangle2_width,
                            s.layer_60_length_2
                        )
        jj_2 = draw.rotate(jj2, 270, origin=(0, 0))
        jj_2_layer60 = draw.rotate(jj2_layer60, 270, origin=(0, 0))
        x_jj = x_a_upper + s.segment_a_width / 2

        jj_1 = draw.translate(
            jj_1,
            xoff=x_jj,
            yoff=y_a_lower
        )

        jj_1_60 = draw.translate(
            jj_1_layer60,
            xoff=x_jj,
            yoff=y_a_lower
        )

        jj_2 = draw.translate(
            jj_2,
            xoff=x_jj,
            yoff=y_a_upper
        )

        jj_2_60 = draw.translate(
            jj_2_layer60,
            xoff=x_jj,
            yoff=y_a_upper
        )

        if mirror:
            # Mirror the SQUID body and the JJ *positions*, but preserve the
            # global +x direction of every JJ shape for foundry consistency.
            squid_no_jj = draw.scale(
                squid_no_jj, xfact=-1, yfact=1, origin=(0, 0)
            )
            # Place the +x-facing pair across the mirrored physical JJ gap,
            # using one shared base offset for layer 20 and layer 60.
            mirrored_base_x = -(x_c - s.segment_c_width / 2)
            junction_xoff = mirrored_base_x - x_jj
            jj_1 = draw.translate(jj_1, xoff=junction_xoff)
            jj_2 = draw.translate(jj_2, xoff=junction_xoff)
            jj_1_60 = draw.translate(jj_1_60, xoff=junction_xoff)
            jj_2_60 = draw.translate(jj_2_60, xoff=junction_xoff)

        design1 = draw.union(squid_no_jj, jj_1, jj_2)
        design_layer60 = draw.union(jj_1_60, jj_2_60)

        #this plate2_anchor will allow us to connect it to the T on the transmon.
        plate2_anchor = draw.Point(
                            x_plate2,
                            y_c
                        )

        if mirror:
            plate2_anchor = draw.scale(
                plate2_anchor,
                xfact=-1,
                yfact=1,
                origin=(0, 0)
            )

        #we rotate it as we are going to the same with the entire design based off user input
        plate2_anchor_rotated = draw.rotate(
                    plate2_anchor,
                    s.rotation,
                    origin=(0, 0)
                )
        x_translation = attach_x - plate2_anchor_rotated.x
        y_translation = attach_y - plate2_anchor_rotated.y

        #This extension plate anchor will act as the coordinates of the second T in the Si:
        extension_plate_anchor = draw.Point(
                            x_extension_plate,
                            y_extension_plate
                        )
        if mirror:
            extension_plate_anchor = draw.scale(
                extension_plate_anchor,
                xfact=-1,
                yfact=1,
                origin=(0, 0)
                    )
            
        extension_plate_anchor_rotated = draw.rotate(
                            extension_plate_anchor,
                            s.rotation,
                            origin=(0, 0)
                        )

        x_coordinate_extension_plate = (
            attach_x
            + extension_plate_anchor_rotated.x
            - plate2_anchor_rotated.x
            )
        
        y_coordinate_extension_plate = (
            attach_y
            + extension_plate_anchor_rotated.y
            - plate2_anchor_rotated.y
        )

        #we rotate the squid if needed
        design1 = draw.rotate(
                    design1,
                    s.rotation,
                    origin=(0, 0)
                )

        #rename the center of  the design based on the location of the pin on plate 2.
        design1 = draw.translate(
                    design1,
                    xoff=x_translation,
                    yoff=y_translation
                )

        #This also includes a rotation of the JJS:
        design_layer60 = draw.rotate(
                    design_layer60,
                    s.rotation,
                    origin=(0, 0)
                )
        design_layer60 = draw.translate(
                    design_layer60,
                    xoff=x_translation,
                    yoff=y_translation
                )
        # SE1JJ is a device-definition layer: the foundry requires every
        # completed polygon vertex on the 5 nm manufacturing grid.
        design_layer60 = draw.shapely.set_precision(design_layer60, 0.000005)
        
        geom = {"design": design1}

    

        self.add_qgeometry("poly", geom, layer=int(p.layer), subtract=False)
        self.add_qgeometry(
            "poly",
            {"jj_layer60": design_layer60},
            layer=60,
            subtract=False
        )
        return x_coordinate_extension_plate, y_coordinate_extension_plate

    def jj_squid(self, sides_lengths, rectangle_width, rectangle_length):
        c = sides_lengths[0]  # base of the triangle
        a = sides_lengths[1]  # side from B to C
        b = sides_lengths[2]  # side from A to C

        # Triangle vertices
        vertex_A = (-c / 2, 0)
        vertex_B = ( c / 2, 0)
        vertex_C_x =  0
        vertex_C_y = np.sqrt(b**2 - (c / 2)**2)

        triangle = draw.Polygon([
            vertex_A,
            vertex_B,
            (vertex_C_x, vertex_C_y)
        ])


        if rectangle_width <= 0:
            raise ValueError("rectangle_width must be positive.")

        if rectangle_width > c:
            raise ValueError(
                "rectangle_width cannot be greater than the triangle base."
            )

        
        width_fraction = rectangle_width / c

        overlap_depth = vertex_C_y * width_fraction

        rectangle_bottom_y = vertex_C_y - overlap_depth

        left_contact_x = (
            vertex_C_x
            + width_fraction * (vertex_A[0] - vertex_C_x)
        )

        right_contact_x = (
            vertex_C_x
            + width_fraction * (vertex_B[0] - vertex_C_x)
        )
        rectangle_center_x = (left_contact_x + right_contact_x) / 2

        rectangle_center_y = rectangle_bottom_y + rectangle_length / 2

        rectangle = draw.rectangle(
            rectangle_width,
            rectangle_length,
            rectangle_center_x,
            rectangle_center_y
        )

        jj = draw.union(triangle, rectangle)

        return jj
    
    def make_t_si(self, p, center_x, center_y):

        t_si_horizontal_local = draw.rectangle(p.t_horizontal_length+p.added_const, p.t_horizontal_width, 0, 0)
        t_si_vertical_local = draw.rectangle(
            p.t_vertical_width,
            p.t_vertical_length,
            -p.t_horizontal_length / 2
            - p.t_vertical_width / 2,
            0
        )

        t_cutout_shape_si_local = draw.union(t_si_horizontal_local,t_si_vertical_local).buffer(0)
        
        if p.transmon_side == -1:
            t_cutout_shape_si_local = draw.scale(
                t_cutout_shape_si_local,
                xfact=-1,
                yfact=1,
                origin=(0, 0)
            )

        local_t_centroid = t_cutout_shape_si_local.centroid
        
        t_cutout_shape_si = draw.translate(
            t_cutout_shape_si_local,
            xoff=center_x - local_t_centroid.x,
            yoff=center_y - local_t_centroid.y
        )

        self.add_qgeometry(
            "poly",
            {"t_si": t_cutout_shape_si},
            layer=int(p.t_layer),
            subtract=True
        )
    def side_arm_placement_and_cutout(self, side_arm, mini_t_horizontal_rectangle, mini_t_vertical_rectangle, p, stem_x, attach_y, stem_width):
            "specifies the two transmon_side of the side arm"
            y_side_arm = attach_y + p.side_arm_move_y
            if p.transmon_side == 1:
                x_side_arm = stem_x - stem_width/2 - p.side_arm_length/2
                x_t_horizontal = x_side_arm - p.side_arm_length/2 + p.t_horizontal_length/2
                x_t_vertical = x_t_horizontal + p.t_horizontal_length/2 + p.t_vertical_width/2
                
            elif p.transmon_side == -1:
                x_side_arm = stem_x + stem_width/2 + p.side_arm_length/2
                x_t_horizontal = x_side_arm + p.side_arm_length/2 - p.t_horizontal_length/2
                x_t_vertical = x_t_horizontal - p.t_horizontal_length/2 - p.t_vertical_width/2
            else:
                raise ValueError("transmon_side must be 1 or -1")
    
            return [draw.translate(side_arm, x_side_arm, y_side_arm), draw.translate(mini_t_horizontal_rectangle, x_t_horizontal, y_side_arm), 
                    draw.translate(mini_t_vertical_rectangle, x_t_vertical, y_side_arm)]

    def make_connection_arm(self, p, stem_x, attach_y, stem_width):

        print("=== ARM DEBUG ===")

        for key in [
            "side_arm_length",
            "side_arm_width",
            "side_arm_move_y",
            "t_horizontal_length",
            "t_horizontal_width",
            "t_vertical_length",
            "t_vertical_width",
            "stem_x",
            "stem_width",
            "attach_y"
        ]:
            value = p.get(key, "MISSING")
            print(key, "=", repr(value), type(value))

            
        side_arm = draw.rectangle(p.side_arm_length, p.side_arm_width, 0, 0)
        mini_t_horizontal_rectangle = draw.rectangle(p.t_horizontal_length, p.t_horizontal_width, 0, 0)
        mini_t_vertical_rectangle = draw.rectangle(p.t_vertical_width, p.t_vertical_length, 0, 0)

        arm, t_horizontal, t_vertical = (
            self.side_arm_placement_and_cutout(
                side_arm,
                mini_t_horizontal_rectangle,
                mini_t_vertical_rectangle,
                p,
                stem_x,
                attach_y,
                stem_width
            )
        )
        t_cutout = draw.union(t_horizontal, t_vertical)

        t_center_x = t_cutout.centroid.x
        t_center_y = t_cutout.centroid.y
        
        pin_half_length = p.t_vertical_width / 2

        squid_pin_points = np.array([
            [t_center_x - pin_half_length, t_center_y],
            [t_center_x + pin_half_length, t_center_y]
        ])

        self.add_pin(
            "squid_center",
            points=squid_pin_points,
            width=p.t_vertical_width
        )

        arm_with_cutout = arm.difference(t_cutout).buffer(0)

        self.add_qgeometry(
            "poly",
            {"connection_arm": arm_with_cutout},
            layer=int(p.t_layer),
            subtract=False
        )

        if p.transmon_side == 1:
            stem_edge_x = stem_x - stem_width / 2
        else:
            stem_edge_x = stem_x + stem_width / 2

        arm_y = attach_y + p.side_arm_move_y

        pin_points = np.array([
            [stem_edge_x, arm_y - p.side_arm_width / 2],
            [stem_edge_x, arm_y + p.side_arm_width / 2]
        ])

        self.add_pin(
            "stem_attach",
            points=pin_points,
            width=p.side_arm_width
        )
        return self.pins["squid_center"]["middle"]

        



In [23]:
# Delete the previous test instance, if present
if "squid_test" in design.components:
    design.delete_component("squid_test")

# Create the SQUID with the corrected class
squid_test = Squid(
    design,
    "squid_test",
    options=Dict(
        stem_x=f"{q1.p.pos_x}mm",
        stem_width=f"{q1.p.trace_width}mm",
        attach_y=f"{q1.p.pos_y + q1.p.squid_offset}mm",
        transmon_side="1"
    )
)

gui.rebuild()
gui.autoscale()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>
=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [24]:
# =====================================
# REFRESH SQUID COMPONENT TEMPLATE
# =====================================

template_key = f"{Squid.__module__}.{Squid.__name__}"

if template_key in design.template_options:
    for key, value in Squid.default_options.items():
        design.template_options[template_key][key] = value

In [25]:
%metal_print TRANSMON

In [26]:
class myTransmon(QComponent):
    """Transmon qubit component: pentagon island, readout lead and an optional SQUID.

    Inherits 'QComponent' class.

    The SQUID branches off the side of the protruding lead ('squid_offset' up the lead from
    the pad), crosses the pocket gap and lands on the ground plane. It is a loop with two
    Josephson-junction arms; each arm has a `squid_jj_gap` break bridged by a junction patch
    on its own layer (`squid_jj_layer`). With orientation 90 the lead points up and
    squid_side='left' puts the SQUID on the qubit's left.

    Default Options:
        * trace_width: width of the readout lead
        * lead_length: length of the readout lead
        * pad_side_length: side length of the pentagon pad
        * pad_gap: gap between pad / lead and ground (pocket)
        * squid_*: SQUID geometry (see below)
    """

    default_options = Dict(
        trace_width="25um",
        lead_length="350um",
        pad_side_length="100um",
        pad_gap="58um",
        orientation="90",
        # --- SQUID on the lead
        squid="False",                 # False: no SQUID
        squid_offset="150um",         # distance from the pad (start of the lead) to the SQUID centre
        squid_side="left",            # 'left' or 'right' of the lead (orientation 90)
        squid_loop_width="20um",      # outer width of the loop, along the lead
        squid_loop_height="20um",     # inner height of the loop, across the gap
        squid_wire_width="3um",       # stems, bars and arms
        squid_jj_gap="2um",           # break in each arm, bridged by a junction patch
        squid_ground_overlap="3um",   # how far the stem reaches into the ground plane
        squid_jj_layer="6",           # layer of the junction patches
    )

    def make(self):
        """Build the qgeometry for the component."""
        p = self.p
        s = p.pad_side_length
        w = p.trace_width
        w_half = w / 2.0
        lead_length = p.lead_length
        pad_gap = p.pad_gap

        #########################################################
        # Pentagon geometry, built with the lead along +x and the pentagon
        # body toward -x. The orientation (default 90) rotates it to point up.
        #
        # The top sides keep the regular-pentagon angle of 36 degrees from the
        # flat top. Their inner ends sit at y = +/- w/2, so the lead fits
        # between them without cutting into them.
        top_angle = np.deg2rad(36.0)
        h1 = s * np.sin(top_angle)                # depth of the shoulders below the top
        y_shoulder = w_half + s * np.cos(top_angle)  # half-width at the shoulders

        # The lower sides (length s) run from the shoulders to a base of length s
        dy_lower = y_shoulder - s / 2.0
        h2 = np.sqrt(s**2 - dy_lower**2)          # depth of the lower sides
        H = h1 + h2                               # total pentagon depth

        # Pentagon vertices (counterclockwise)
        top_pos = (0, w_half)
        shoulder_pos = (-h1, y_shoulder)
        base_pos = (-H, s / 2.0)
        base_neg = (-H, -s / 2.0)
        shoulder_neg = (-h1, -y_shoulder)
        top_neg = (0, -w_half)

        # Main pad: pentagon plus the lead
        transmon = draw.Polygon(
            [
                top_pos,
                shoulder_pos,
                base_pos,
                base_neg,
                shoulder_neg,
                top_neg,
                (lead_length, -w_half),
                (lead_length, w_half),
            ]
        )

        # Pocket: the pad grown by pad_gap (sharp corners), trimmed flush with
        # the end of the lead so the readout pin stays open for routing.
        margin = 2 * pad_gap
        pocket = transmon.buffer(pad_gap, join_style=JOIN_STYLE.mitre)
        pocket = pocket.intersection(
            box(-(H + margin), -(y_shoulder + margin), lead_length, y_shoulder + margin)
        )

        #########################################################
        # SQUID branching off the side of the lead (local frame, before rotation)
        use_squid = str(p.squid).lower() in ("true", "1", "yes")
        if use_squid:
            if p.squid_side not in ("left", "right"):
                raise ValueError("squid_side must be 'left' or 'right'.")
            half_loop = p.squid_loop_width / 2.0
            if not (half_loop < p.squid_offset < lead_length - half_loop):
                raise ValueError("squid_offset puts the SQUID off the end of the lead.")
            island_pts, ground_pts, jj_pts = _squid_outlines(
                w, pad_gap, p.squid_offset, p.squid_side, p.squid_loop_width,
                p.squid_loop_height, p.squid_wire_width, p.squid_jj_gap,
                p.squid_ground_overlap)
            squid_island = draw.Polygon(island_pts)
            squid_ground = draw.Polygon(ground_pts)
            squid_jj_a, squid_jj_b = draw.Polygon(jj_pts[0]), draw.Polygon(jj_pts[1])

        #########################################################
        # Pin lines. Each side is listed clockwise around the pad, so every pin
        # normal points outward, away from the pad.
        main_pin_line = draw.LineString([(lead_length, w_half), (lead_length, -w_half)])

        # Pin names describe positions with the default orientation of 90
        pin_upper_left = draw.LineString([shoulder_pos, top_pos])
        pin_left = draw.LineString([base_pos, shoulder_pos])
        pin_bottom = draw.LineString([base_neg, base_pos])
        pin_right = draw.LineString([shoulder_neg, base_neg])
        pin_upper_right = draw.LineString([top_neg, shoulder_neg])

        polys = [
            main_pin_line, transmon, pocket,
            pin_upper_left, pin_left, pin_bottom, pin_right, pin_upper_right,
        ]
        if use_squid:
            polys += [squid_island, squid_ground, squid_jj_a, squid_jj_b]

        # Rotate and translate everything
        polys = draw.rotate(polys, p.orientation, origin=(0, 0))
        polys = draw.translate(polys, xoff=p.pos_x, yoff=p.pos_y)
        [
            main_pin_line, transmon, pocket,
            pin_upper_left, pin_left, pin_bottom, pin_right, pin_upper_right,
        ] = polys[:8]
        if use_squid:
            squid_island, squid_ground, squid_jj_a, squid_jj_b = polys[8:]

        # Add to the qgeometry table
        self.add_qgeometry("poly", dict(transmon=transmon), layer=p.layer)
        self.add_qgeometry("poly", dict(pocket=pocket), layer=p.layer, subtract=True)
        if use_squid:
            self.add_qgeometry("poly", dict(squid_island=squid_island,
                                            squid_ground=squid_ground), layer=p.layer)
            self.add_qgeometry("poly", dict(squid_jj_a=squid_jj_a, squid_jj_b=squid_jj_b),
                               layer=p.squid_jj_layer)

        # Pins
        self.add_pin("readout", points=list(main_pin_line.coords),
                     width=w, input_as_norm=False)
        self.add_pin("upper_left", points=list(pin_upper_left.coords),
                     width=w, input_as_norm=False)
        self.add_pin("left", points=list(pin_left.coords),
                     width=w, input_as_norm=False)
        self.add_pin("bottom", points=list(pin_bottom.coords),
                     width=w, input_as_norm=False)
        self.add_pin("right", points=list(pin_right.coords),
                     width=w, input_as_norm=False)
        self.add_pin("upper_right", points=list(pin_upper_right.coords),
                     width=w, input_as_norm=False)

In [27]:
%metal_print LAUNCHPADS

In [28]:
# create wirebond pads:
# need to change based on LL rules (page 38, section 4.4)
# wirebond pad location is determined by the tip.
def create_wirebond(name, orientation, pos_y, pos_x, pad_width='150um', pad_length='100um', taper_height='200um', trace_width="5um",trace_gap="3um", layer="5"):
    options = Dict(
        orientation=orientation,
        pos_y=f"{pos_y}mm",
        pos_x=f"{pos_x}mm",
        pad_width=pad_width,
        # LaunchpadWirebond calls the straight rectangular length pad_height.
        pad_height=pad_length,
        taper_height=taper_height,
        trace_width=trace_width,
        trace_gap=trace_gap,
        layer=layer

    )
    return LaunchpadWirebond(design, name, options=options)

In [29]:
#Page 38, Section 4.4. Launchpads must be 200 um away from chip edge
#Only the launchpads this chip uses; names and positions are the same as in
#the full 24-pad layout (pad_i_x on the top/bottom edges, pad_i_y on the sides).
#create_wirebond(name, orientation, pos_y, pos_x)

#READOUT LINE: 10um trace / 6um gap tip to match the readout line (a 5/3 tip
#leaves a 0.5um ground notch where they join, M1.S.1)
pad_4_y = create_wirebond("pad_4_y", 0, 2, -2.0979, trace_width="10um", trace_gap="6um")
pad_10_y = create_wirebond("pad_10_y", 180, 2, 2.0979, trace_width="10um", trace_gap="6um")

#CHARGE LINES:
pad_6_y = create_wirebond("pad_6_y", 0, 0.4, -2.0979)
#pad_12_y = create_wirebond("pad_12_y", 180, 0.4, 2.0979)

#FLUX LINES:
pad_3_y = create_wirebond("pad_3_y", 0, 1.2, -2.0979)   # charge line (top pad)


pad_9_y = create_wirebond("pad_9_y", 180, 1.2, 2.0979, trace_width="10um", trace_gap="6um")   # charge line (top pad)

#WAVEGUIDE: on the left and right edges at the waveguide's height
#(y = -0.0817mm), so the feeds run straight in to the qubits and the lower half
#of the chip is free for the waveguide's serpentines; 10/6 tip like the readout
#pad_wg_in = create_wirebond("pad_wg_in", 0, -0.0817, -2.0979, trace_width="10um", trace_gap="6um")
#pad_wg_out = create_wirebond("pad_wg_out", 180, -0.0817, 2.0979, trace_width="10um", trace_gap="6um")

gui.rebuild()
gui.autoscale()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [30]:
%metal_print CHARGE LINE - HAVE TO UPDATE

In [31]:
# CHARGE LINES: from launchpads at y = 1.2mm (pad_3_y on the left to an open end
# beside qubit's TOP pad. The bottom pad's sides belong to the waveguide's jogs, and
# a line from below would have to cross the waveguide's feed. The routes come
# down at x = -0.5mm, clear of the Purcell filters' jogs; the open end sits at y = -0.4mm

# LEFT:
open_charge_a = OpenToGround(
    design=design,
    name="open_charge",
    options=Dict(
        pos_x="-0.3mm",
        pos_y="-.4mm",
        orientation="0",
        width="5um",
        gap="3um",
        termination_gap="6um",
        layer="5",
    )
)

charge_line_a = RouteAnchors(
    design=design,
    name="charge_line",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component="pad_3_y",
                pin="tie"
            ),
            end_pin=Dict(
                component="open_charge",
                pin="open"
            )
        ),
        anchors=OrderedDict({
            0: np.array([-0.5, 1.2]),
            1: np.array([-0.5, -.4]),
        }),
        fillet="50um",
        trace_width="5um",
        trace_gap="3um",
        layer="5",
    )
)


gui.rebuild()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [32]:
%metal_print FLUX LINE - HAVE TO UPDATE


In [33]:
print("SQUID attachment:", squid_test.pins["squid_center"]["middle"])
print("Extension:", squid_test.extension_plate_center)

SQUID attachment: [-0.0625 -0.45  ]
Extension: (np.float64(-0.085), np.float64(-0.464))


In [ ]:
# FLUX LINES, like Ding et al., PRX 13, 031035 (2023): plain CPWs from the
# upper side launchpads (pad_6_y on the left, pad_12_y on the right, y = 0.4mm)
# to the qubit's "flux" pin, on the gap's centre line (y = 0). From there the
# qubit draws the PRX-style end: the trace narrows to 3um, the gaps flare out
# to 41um, and the trace is shorted to the ground strip at the edge of the
# loop's bay, so the current runs along that edge past the loop. The routes
# turn down at x = +/-1.5mm; the waveguide passes 82um below the line's end.
# As in the paper, the static Phi0/2 bias would come from a global coil and
# these lines do fine tuning; the mutual inductance needs a simulation.

# LEFT:
# squid_ref_x, squid_ref_y = squid_test.pins["squid_center"]["middle"]

# flux_line_end = OpenToGround(
#     design=design,
#     name="squid_flux_end",
#     options=Dict(
#         pos_x=f"{squid_ref_x - 0.05}mm",
#         pos_y=f"{squid_ref_y + 0.04}mm",
#         orientation="0",
#         width="5um",
#         gap="3um",
#         termination_gap="6um",
#         layer="5",
#     )
# )


# flux_line_a = RouteAnchors(
#     design=design,
#     name="flux_line_squid",
#     options=Dict(
#         pin_inputs=Dict(
#             start_pin=Dict(
#                 component="pad_6_y",
#                 pin="tie"
#             ),
#             end_pin=Dict(
#                 component="squid_flux_end",
#                 pin="open"
#             )
#         ),

#         anchors=OrderedDict({
#             0: np.array([-1.8,  0.4]),
#     1: np.array([-1.8, -0.2]),
#     2: np.array([-0.8, -0.2]),
#     3: np.array([-0.8, -0.4])
#         }),

#         fillet="50um",
#         trace_width="5um",
#         trace_gap="3um",
#         layer="5",
#     )
# )

# gui.rebuild()
# gui.autoscale()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [35]:
%metal_print READOUT LINE

In [36]:
# READOUT LINE

readout_line = RouteStraight(
    design=design,
    name="readout_line",
    options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(
                component="pad_4_y",
                pin="tie"
            ),
            end_pin=Dict(
                component="pad_10_y",
                pin="tie"
            ),
        ),
        advanced=Dict(
            avoid_collision=False
        ),
        trace_width="10um",
        trace_gap="6um",
        fillet="50um",
        layer="5",
    )
)

gui.rebuild()
gui.autoscale()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [37]:
%metal_print PURCELL FILTER AND READOUT RESONATOR - VERIFY LENGTHS

In [38]:
# purcell filter and readout resonator for qubit
#Purcell filter and readout resonator for the qubits: CandyCanePurcell
#(Val's straight version, no meanders). Each line is one straight run with a
#single U-turn. Note the component's path names are the other way round from
#their job here:
#  "resonator" path = Purcell filter: from the T pad on the readout line,
#                     down, U-turn, back up (on the chip-centre side)
#  "purcell" path   = readout resonator: from resonator_connect (routed to the
#                     qubit's readout coupler), up, U-turn, down (chip-edge side)
#Frequencies from Almanakly et al., "Driven giant atoms" (Table S6): qubit A's
#resonator and filter at 9.8 GHz, qubit B's at 9.9 GHz. The resonators are
#lambda/4 by Val's meander_length formula (3.03 and 3.00 mm); the filters are
#shorter (2.42 and 2.39 mm drawn) because of the capacitance of their pad by
#the readout line, as in the paper's device (see the tuner cell). The tuner
#sets up_from_turn (filter) and down_from_turn (resonator). fillet = 70um makes each U-turn 140um wide. No meanders, like
#the paper's (Fig. 2a): the resonator goes straight up from the qubit, turns
#under the readout line and comes straight back down (ends at y ~ 0.72mm);
#the filter goes straight down from its T pad, turns at y ~ 0.30mm, just above
#the qubit, and comes back up to y ~ 1.1mm. As in the paper, the resonator's hairpin is on the chip-edge
#side and the filter's (with its T pad) on the chip-centre side.
#(CandyCanePurcellMeander can add jogs: meander_jogs / filter_meander_jogs,
#both 0 here.)
#pos_x (the T pad) = qubit_x - 2*89.282um - 10.536um = 0.6327mm (towards the
#chip centre) puts resonator_connect right above each qubit's readout coupler;
#orientation = 1 for qubit A and -1 for qubit B puts the filter on the inside.

class CandyCanePurcellMeander(CandyCanePurcell):
    """CandyCanePurcell with meander jogs on the return leg of each path.
    "purcell" path (the readout resonator here): meander_jogs on its leg back
    down, towards -direction, starting meander_start below the U-turn.
    "resonator" path (the Purcell filter here): filter_meander_jogs on its leg
    back up, towards +direction, starting
    filter_meander_start above the U-turn. Each jog goes meander_width
    (filter: filter_meander_width) sideways, meander_depth along the leg and
    back; jogs are meander_gap apart. With fillet = 70um, widths, depth and
    gap must be >= 140um."""
    default_options = Dict(
        meander_jogs="0",
        filter_meander_jogs="0",
        meander_width="140um",
        filter_meander_width="140um",
        meander_depth="140um",
        meander_gap="140um",
        meander_start="500um",
        filter_meander_start="600um",
    )

    def purcell(self, start_pin_x, start_pin_y, p, direction):
        # rounded_centerline (below) adds the jogs while this path is drawn
        self._meander_for = ("purcell", p, direction)
        try:
            return super().purcell(start_pin_x, start_pin_y, p, direction)
        finally:
            self._meander_for = None

    def resonator(self, start_pin_x, start_pin_y, purcell_rect_x, purcell_rect_y, p, direction):
        self._meander_for = ("resonator", p, direction)
        try:
            return super().resonator(start_pin_x, start_pin_y, purcell_rect_x,
                                     purcell_rect_y, p, direction)
        finally:
            self._meander_for = None

    def rounded_centerline(self, anchors, radius, *args, **kwargs):
        if getattr(self, "_meander_for", None):
            path, p, direction = self._meander_for
            if path == "purcell":
                anchors = self.add_meander(anchors, p, int(p.meander_jogs), -direction,
                                           -1, p.meander_start, p.meander_width)
                # the airbridges follow these points
                self._purcell_points = self.rotate_points(
                    np.array(list(anchors.values()), dtype=float), p)
            else:
                anchors = self.add_meander(anchors, p, int(p.filter_meander_jogs), direction,
                                           1, p.filter_meander_start, p.filter_meander_width)
                self._resonator_points = self.rotate_points(
                    np.array(list(anchors.values()), dtype=float), p)
        return super().rounded_centerline(anchors, radius, *args, **kwargs)

    def add_meander(self, anchors, p, jogs, side, along, start, width):
        """Insert jogs on the leg from anchor 2 to anchor 3; along = -1 if
        that leg runs down, +1 if it runs up; side = direction of the jogs."""
        pts = [np.array(v, dtype=float) for v in anchors.values()]
        if jogs <= 0:
            return anchors
        x_leg, y = pts[2]
        new = pts[:3]
        y += along * start
        for _ in range(jogs):
            y_next = y + along * p.meander_depth
            new += [np.array([x_leg, y]),
                    np.array([x_leg + side*width, y]),
                    np.array([x_leg + side*width, y_next]),
                    np.array([x_leg, y_next])]
            y = y_next + along * p.meander_gap
        new.append(pts[3])
        return OrderedDict(enumerate(new))




In [39]:
purcell_and_res1 = CandyCanePurcellMeander(
    design=design,
    name="purcell_and_resonator_1",
    options=Dict(
        pos_x="0.189mm",
        pos_y="2.005mm",
        orientation="1",
        fillet="70um",
        meander_line_width="10um",
        si_fill="4um",
        down_from_pin_start="1522.8um",  # filter down to y ~ 0.30mm
        up_from_turn="812.3um",          # filter back up (tuned in the next cell)
        up_from_pin_start="880um",       # resonator_connect at y ~ 1.0mm
        down_from_turn="1164.6um",       # resonator back down (tuned in the next cell)
        meander_jogs="0",
        filter_meander_jogs="0",
    )
)

gui.rebuild()
gui.autoscale()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


In [40]:
%metal_print WAVEGUIDE COUPLING

In [41]:
import math
import numpy as np
from scipy.spatial import cKDTree
from qiskit_metal import draw, Dict
from qiskit_metal.qlibrary.core import QComponent

ARC_POINTS = 33   # points per rounded corner


# ----------------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------------
def _fillet_path(points, r, n_arc=ARC_POINTS, tol=1e-9):
    """Round every corner of a polyline with a circular arc of radius r (any angle).

    Raises if an arc does not fit between its neighbours (nothing is silently left
    sharp or allowed to overlap)."""
    # points are (x, y) or (x, y, radius); a missing radius means `r`
    arr = np.array([[q[0], q[1], q[2] if len(q) > 2 else r] for q in points], dtype=float)
    keep = [0] + [i for i in range(1, len(arr)) if np.linalg.norm(arr[i, :2] - arr[i - 1, :2]) > tol]
    arr = arr[keep]
    pts, rad = arr[:, :2], arr[:, 2]
    d = np.diff(pts, axis=0)
    seg = np.linalg.norm(d, axis=1)
    u = d / seg[:, None]
    n = len(pts)
    phi = np.zeros(n)          # turning angle at each vertex
    t = np.zeros(n)            # tangent length of the arc at each vertex
    for i in range(1, n - 1):
        phi[i] = np.arccos(np.clip(u[i - 1] @ u[i], -1.0, 1.0))
        if phi[i] > np.pi - 1e-6:
            raise ValueError(f"Path reverses on itself at {tuple(pts[i])}.")
        if phi[i] > 1e-9:
            t[i] = rad[i] * np.tan(phi[i] / 2)
    short = np.where(seg < t[:-1] + t[1:] - 1e-9)[0]
    if len(short):
        i = int(short[0])
        raise ValueError(f"Segment {tuple(pts[i])} -> {tuple(pts[i + 1])} is shorter than the "
                         "fillets at its ends need. Reduce `fillet` or lengthen the "
                         "routing distances (pitch, leg_length, bump_height, ...).")
    out = [pts[0]]
    for i in range(1, n - 1):
        if phi[i] <= 1e-9:
            out.append(pts[i])
            continue
        p1 = pts[i] - u[i - 1] * t[i]
        s = np.sign(u[i - 1][0] * u[i][1] - u[i - 1][1] * u[i][0])
        c = p1 + rad[i] * s * np.array([-u[i - 1][1], u[i - 1][0]])
        a0 = np.arctan2(p1[1] - c[1], p1[0] - c[0])
        a = a0 + s * np.linspace(0.0, phi[i], n_arc)
        out.extend(c + rad[i] * np.column_stack([np.cos(a), np.sin(a)]))
    out.append(pts[-1])
    return np.array(out)


def _arclength(coords):
    seg = np.linalg.norm(np.diff(coords, axis=0), axis=1)
    return np.concatenate([[0.0], np.cumsum(seg)])


def _s_at_point(coords, pt):
    """Arc-length position of the point on the polyline closest to `pt`."""
    pt = np.asarray(pt, dtype=float)
    cum = _arclength(coords)
    best_d, best_s = np.inf, 0.0
    for k in range(len(coords) - 1):
        a, ab = coords[k], coords[k + 1] - coords[k]
        L2 = ab @ ab
        t = 0.0 if L2 == 0 else float(np.clip((pt - a) @ ab / L2, 0.0, 1.0))
        d = np.linalg.norm(a + t * ab - pt)
        if d < best_d:
            best_d, best_s = d, cum[k] + t * np.sqrt(L2)
    return best_s


def _turn_t(r, d_in, d_out):
    """Tangent length of a radius-r fillet between two unit directions."""
    phi = np.arccos(np.clip(np.dot(d_in, d_out), -1.0, 1.0))
    return r * np.tan(phi / 2)


def _self_clearance(coords, min_dist, step=0.0025):
    """Smallest distance between parts of the path that are far apart along the path."""
    cum = _arclength(coords)
    s = np.arange(0.0, cum[-1], step)
    xy = np.column_stack([np.interp(s, cum, coords[:, 0]), np.interp(s, cum, coords[:, 1])])
    tree = cKDTree(xy)
    worst = np.inf
    skip = 3 * min_dist                                  # neighbours along the path don't count
    for i, j in tree.query_pairs(min_dist, output_type="ndarray"):
        if abs(s[j] - s[i]) > skip:
            worst = min(worst, float(np.linalg.norm(xy[i] - xy[j])))
    return worst


def _rows(y_first, n, pitch, x_left, x_right, x_final):
    """Vertices for n horizontal rows stacked downward from y_first.
    Row 0 heads right from the vertex the caller already added; n must be even so the
    last row heads left, ending at x_final."""
    v = []
    for k in range(n):
        y = y_first - k * pitch
        x = x_right if k % 2 == 0 else (x_left if k < n - 1 else x_final)
        v.append((x, y))
        if k < n - 1:
            v.append((x, y - pitch))
    return v


def _layout(g, n, WA, WB):
    """Waypoints of the whole waveguide (corner vertices, before filleting). Right half:

      chip edge (top) -> along the top -> down a lane parallel to lane x_A -> in along a
      row parallel to the row leaving run 1 -> leg perpendicular to the upper-right side ->
      90 deg corner onto that side (run 1) -> 90 deg corner off it -> perpendicular leg ->
      row out to lane x_A -> up to block A (upper right) -> rows, the last one runs back in
      to a leg perpendicular to the lower-right side -> 90 deg corner onto it (run 2) ->
      90 deg corner off it -> leg -> row 0 of block B (lower right) -> rows -> lane x_B ->
      feeder under the qubit -> bump up to the bottom run (run 3).

    The left half is the mirror image, traversed in reverse; it stops (open) at the end
    of the left slant run."""
    pitch, qx, r, rc = g["pitch"], g["qx"], g["r"], g["rc"]
    x_Al = x_Bl = g["x_blk"]
    x_A, x_B = x_Al - pitch, x_Bl - pitch
    x_f = x_A - g["feed_spacing"]                          # feed lane, parallel to lane x_A
    y_top = g["Qin"][1] + (n - 1) * pitch                  # first row of block A
    y_feed = g["start_y"] if g["start_y"] is not None else y_top + g["feed_clear"]

    R = [(g["start_x"], y_feed), (x_f, y_feed, r), (x_f, g["yf2"], r), (*g["Q"], r),
         (*g["W1"], rc), (*g["W2"], rc), (*g["Q2"], r), (x_A, g["Q2"][1], r), (x_A, y_top, r)]
    R += _rows(y_top, n, pitch, x_Al, x_Al + WA, g["Qin"][0])   # last vertex is Qin
    R += [(*g["Wa"], rc), (*g["Wb"], rc), (*g["Qout"], r)]
    R += _rows(g["Qout"][1], n, pitch, x_Bl, x_Bl + WB, x_B)
    R += [(x_B, g["y_h"], r), (g["x_p3"], g["y_h"], rc), (g["x_p3"], g["y_b"], rc), (qx, g["y_b"])]
    M = [(2 * qx - q[0], q[1], *q[2:]) for q in reversed(R)][1:]
    # left end: stop at the end of the left slant run (drop the mirrored feed vertices)
    M = M[:-5] + [(2 * qx - g["P1"][0], g["P1"][1])]
    return R + M


class GiantAtomWaveguide(QComponent):
    """Giant-atom waveguide capacitively coupled to all five sides of the pentagon transmon
    (`myTransmon`).

    Qubit orientation 90 (lead up). Coupled sides, in the order the waveguide meets them:

        upper-right side -> lower-right side -> base -> lower-left side -> upper-left side

    On each side the waveguide runs parallel to the metal for `coupling_length` at a fixed
    edge-to-edge distance `coupling_gap`; every run is centred on its side. Between
    consecutive runs the waveguide has `target_length` (run centre to run centre), taken up
    by a serpentine of horizontal rows.

    Layout: the four serpentine blocks sit beside the qubit, one above and one below the
    qubit's level on each side. The right end of the waveguide comes in along the top of
    the chip to `start_x`; the left end is open and stops right after the left slant run.
    The layout is mirror-symmetric about x = qubit_x. Bends have radius `fillet`, except the 90 deg corners right next to the coupled runs
    (radius `coupling_fillet`). Every run is entered and left at 90 deg through a leg
    perpendicular to its side.
    """

    default_options = Dict(
        # --- qubit geometry (match the myTransmon options)
        qubit_x="0mm",
        qubit_y="-0.6mm",             # y of the neck/pad junction (myTransmon pos_y)
        qubit_side_length="100um",   # myTransmon pad_side_length
        qubit_trace_width="25um",    # myTransmon trace_width (neck width)
        qubit_lead_length="350um",   # myTransmon lead_length (only used for the clearance check)
        # --- coupling
        coupling_gap="10um",         # edge-to-edge distance, waveguide to qubit metal
        coupling_length="40um",      # length of every coupled run (centred on its side)
        leg_length="100um",          # legs perpendicular to the sides, leading into / out of the runs
        coupling_fillet="30um",      # radius of the 90 deg corners right next to the coupled runs
        bump_height="100um",         # height of the bump that carries the bottom run (>= 2*coupling_fillet)
        feed_spacing="100um",        # feed line runs this far from the (parallel) line under it
        # --- serpentine blocks (x measured from qubit_x)
        block_x_min="0.75mm",        # left turns of all four blocks (closest to the qubit)
        serp_x_max="2mm",          # right-most x of the blocks = start of the launchpad
        feed_clearance="150um",      # feed line runs this far above the top row of block A (only used if start_y is None)
        y_max="2.35mm",              # highest the feed line may reach
        y_min="-2.35mm",             # lowest the lower blocks may reach
        target_length="22mm",        # run centre -> next run centre
        rows=0,                      # 0 = auto, else force an even row count
        pitch="100um",               # row spacing, must be >= 2*fillet
        fillet="50um",               # radius of all waveguide bends except those at the coupled runs
        # --- chip / layout
        start_x="2.0979mm",             # chip edge where the right end terminates
        start_y="1.2mm",                # y of the feed line / start of the waveguide (None = feed_clearance above block A)
        trace_width="10um",
        trace_gap="6um",
        layer="5",
        chip="main",
    )
    component_metadata = Dict(short_name="waveguide", _qgeometry_table_path="True")

    def get_points(self):
        return self._waveguide_points

    def make(self):
        p = self.p
        r, w, pitch = p.fillet, p.trace_width, p.pitch
        qx, qy, L, tw = p.qubit_x, p.qubit_y, p.qubit_side_length, p.qubit_trace_width
        Lc, rc = p.coupling_length, p.coupling_fillet
        if pitch < 2 * r - 1e-12:
            raise ValueError("pitch must be >= 2*fillet")
        if p.bump_height < 2 * rc - 1e-12:
            raise ValueError("bump_height must be at least 2*coupling_fillet")
        if p.rows and (int(p.rows) != p.rows or int(p.rows) % 2):
            raise ValueError("rows must be 0 (auto) or an even integer")
        if not 0 < Lc < L:
            raise ValueError("coupling_length must be positive and shorter than the side length")

        # ---- pentagon (right half), same construction as myTransmon (orientation 90)
        a = tw / 2.0
        top_angle = np.deg2rad(36.0)
        h1 = L * np.sin(top_angle)                       # depth of the shoulders below the neck
        ys = a + L * np.cos(top_angle)                   # half-width at the shoulders
        dy = ys - L / 2.0
        h2 = math.sqrt(L ** 2 - dy ** 2)                 # depth of the lower sides
        H = h1 + h2                                      # total depth of the pentagon

        D = p.coupling_gap + w / 2.0                     # centreline-to-metal distance
        neck = np.array([qx + a, qy])
        shoulder = np.array([qx + ys, qy - h1])
        base = np.array([qx + L / 2.0, qy - H])

        x_hat = np.array([1.0, 0.0])
        u1 = (shoulder - neck) / L                       # upper side, neck -> shoulder
        n1 = np.array([-u1[1], u1[0]])                   # outward normal (up / right)
        u2 = (base - shoulder) / L                       # lower side, shoulder -> base
        n2 = np.array([-u2[1], u2[0]])                   # outward normal (right / down)

        s0, s1 = (L - Lc) / 2.0, (L + Lc) / 2.0          # run start / end along each side
        O1, O2 = neck + D * n1, shoulder + D * n2

        # every run is entered and left through a 90 deg corner (radius rc) from a leg
        # perpendicular to its side
        leg = p.leg_length
        # run 1 (upper side)
        P1, P2 = O1 + s0 * u1, O1 + s1 * u1
        W1, W2 = P1 - rc * u1, P2 + rc * u1
        Q2 = W2 + leg * n1                               # end of the exit leg -> row heading right
        # the feed comes in along a row `feed_spacing` above the row leaving run 1 and
        # meets the entry leg (the line through W1 along n1) at Q
        yf2 = Q2[1] + p.feed_spacing
        Q = W1 + (yf2 - W1[1]) / n1[1] * n1
        # run 2 (lower side)
        Pa, Pb = O2 + s0 * u2, O2 + s1 * u2
        Wa, Wb = Pa - rc * u2, Pb + rc * u2
        Qin = Wa + leg * n2                              # row heading left -> entry leg
        Qout = Wb + leg * n2                             # exit leg -> row heading right
        # run 3 (base) with the bump
        y_b = qy - H - D
        y_h = y_b - p.bump_height
        x_p3 = qx + Lc / 2.0 + rc
        x_blk = qx + p.block_x_min

        sp = w + p.trace_gap
        if p.feed_spacing < sp:
            raise ValueError("feed_spacing must be at least trace_width + trace_gap")
        if Q2[1] - Qin[1] < sp:
            raise ValueError("The row leaving the upper-right run and the row entering the "
                             "lower-right run are too close; increase leg_length.")
        if Qout[1] - y_h < sp + rc:
            raise ValueError("Feeder under the qubit is too close to the first row of the "
                             "lower blocks; increase bump_height.")

        g = dict(qx=qx, start_x=p.start_x, start_y=p.start_y, pitch=pitch, r=r, rc=rc, P1=tuple(P1),
                 Q=tuple(Q), yf2=yf2, feed_spacing=p.feed_spacing, W1=tuple(W1), W2=tuple(W2), Q2=tuple(Q2), Wa=tuple(Wa),
                 Wb=tuple(Wb), Qin=tuple(Qin), Qout=tuple(Qout), y_b=y_b, y_h=y_h,
                 x_p3=x_p3, x_blk=x_blk, feed_clear=p.feed_clearance)

        # coupled-run centres: upper R, lower R, base, lower L, upper L
        c_1 = tuple(O1 + (L / 2.0) * u1)
        c_2 = tuple(O2 + (L / 2.0) * u2)
        c_3 = (qx, y_b)
        centres = [c_1, c_2, c_3, (2 * qx - c_2[0], c_2[1]), (2 * qx - c_1[0], c_1[1])]

        # ---- size the serpentine blocks. All blocks get the same row count n; length is
        # linear in the row width (slope n), so probe at W = 1 mm and solve in closed form.
        W_cap = p.serp_x_max - p.block_x_min

        def link_length(n, WA, WB, i):
            coords = _fillet_path(_layout(g, n, WA, WB), r)
            s = [_s_at_point(coords, c) for c in centres[i:i + 2]]
            return s[1] - s[0]

        n_found = None
        for n in ([int(p.rows)] if p.rows else range(2, 400, 2)):
            try:
                WB = 1.0 + (p.target_length - link_length(n, 1.0, 1.0, 1)) / n
            except ValueError:
                continue                       # too few rows for the routing to fit
            if WB <= W_cap:
                n_found = n
                break
        if n_found is None:
            raise ValueError(f"The blocks do not fit: row width would exceed the {W_cap:.3f} mm "
                             "available. Increase serp_x_max or reduce block_x_min.")
        n = n_found
        WA = 1.0 + (p.target_length - link_length(n, 1.0, WB, 0)) / n
        if min(WA, WB) < 2 * r:
            raise ValueError("Serpentine rows would be shorter than 2*fillet.")
        if WA > W_cap + 1e-9:
            raise ValueError(f"Block A needs row width {WA:.3f} mm, above the {W_cap:.3f} mm "
                             "available; increase serp_x_max.")
        y_top = Qin[1] + (n - 1) * pitch
        y_feed = p.start_y if p.start_y is not None else y_top + p.feed_clearance
        if y_feed - y_top < sp + r:
            raise ValueError(f"start_y = {y_feed:.3f} mm leaves only {(y_feed - y_top) * 1e3:.0f} um "
                             f"above the top row of block A (y = {y_top:.3f} mm); at least "
                             f"{(sp + r) * 1e3:.0f} um is needed. Raise start_y, or use a smaller "
                             "pitch / wider blocks so fewer rows are needed.")
        y_last = Qout[1] - (n - 1) * pitch
        if y_feed > p.y_max:
            raise ValueError(f"The feed line would sit at y = {y_feed:.3f} mm, above y_max = "
                             f"{p.y_max:.3f} mm. Use a smaller pitch or wider blocks.")
        if y_last < p.y_min:
            raise ValueError(f"Block B would reach y = {y_last:.3f} mm, below y_min = "
                             f"{p.y_min:.3f} mm. Use a smaller pitch or wider blocks.")

        points = _layout(g, n, WA, WB)
        coords = _fillet_path(points, r)
        cum = _arclength(coords)

        # nothing may come closer to the qubit metal than the coupling distance
        lead = p.qubit_lead_length
        pad = draw.Polygon([(qx - a, qy + lead), (qx + a, qy + lead), (qx + a, qy),
                            tuple(shoulder), tuple(base), (2 * qx - base[0], base[1]),
                            (2 * qx - shoulder[0], shoulder[1]), (qx - a, qy)])
        dmin = draw.LineString(coords).distance(pad)
        if dmin < D - 1e-4:
            raise ValueError(f"The waveguide comes within {(dmin - w / 2) * 1e3:.1f} um of the "
                             f"qubit metal (coupling_gap = {p.coupling_gap * 1e3:.1f} um) away "
                             "from the coupled runs. Reduce coupling_length.")
        # and no two separate parts of the waveguide may come closer than trace_width + gap
        worst = _self_clearance(coords, sp - 1e-4)
        if worst < sp - 1e-4:
            raise ValueError(f"Two parts of the waveguide come within {worst * 1e3:.1f} um "
                             f"(centre to centre) of each other; {sp * 1e3:.1f} um is needed. "
                             "Reduce coupling_length or change feed_spacing / bump_height.")
        s_c = [_s_at_point(coords, c) for c in centres]

        # bookkeeping to inspect from the notebook
        self._waveguide_points = np.array([q[:2] for q in points], dtype=float)
        self.coupling_centres = centres
        self.coupling_spacings = list(np.diff(s_c))     # run 1->2, 2->3, 3->4, 4->5
        self.coupling_length = Lc
        self.rows_per_block = n
        self.row_width_block_a, self.row_width_block_b = WA, WB
        self.length_total = cum[-1]

        centerline = draw.LineString(coords)
        self.add_qgeometry("path", {"waveguide": centerline},
                           width=w, fillet=0, layer=p.layer, chip=p.chip)
        self.add_qgeometry("path", {"waveguide_etch": centerline},
                           width=w + 2 * p.trace_gap, fillet=0,
                           layer=p.layer, subtract=True, chip=p.chip)

        def pin(name, end, dirn):
            t = np.array([dirn[1], -dirn[0]])
            self.add_pin(name, input_as_norm=False, width=w,
                         points=np.array([end - t * w / 2, end + t * w / 2]))

        xy = self._waveguide_points
        d1 = xy[-1] - xy[-2]
        pin("waveguide_connect", xy[0], (1, 0))                    # right end, at the chip edge
        pin("waveguide_end", xy[-1], d1 / np.linalg.norm(d1))      # left end, open

In [42]:
# instantiation
design.template_options.pop(myTransmon._get_unique_class_name(), None)
design.template_options.pop(GiantAtomWaveguide._get_unique_class_name(), None)

q1 = myTransmon(design, 'Q1', options=dict(pos_x='0um', pos_y='-0.6mm'))
wg = GiantAtomWaveguide(design=design, name="waveguide")

gui.rebuild()
gui.autoscale()

=== ARM DEBUG ===
side_arm_length = 0.0525 <class 'float'>
side_arm_width = 0.01 <class 'float'>
side_arm_move_y = 0.0 <class 'float'>
t_horizontal_length = 0.002 <class 'float'>
t_horizontal_width = 0.002 <class 'float'>
t_vertical_length = 0.006 <class 'float'>
t_vertical_width = 0.002 <class 'float'>
stem_x = 0.0 <class 'float'>
stem_width = 0.025 <class 'float'>
attach_y = -0.44999999999999996 <class 'float'>


11:19PM 04s ERROR [rebuild]: ERROR in building component name=flux_line_squid, error=Qiskit Metal - Designer Error: User action required. 
connect_simple() has failed. This might be due to one of two reasons. 1. Either one of the start point [-0.8 -0.4] or the end point [-0.115 -0.41 ] provided are inside the bounding box of another QComponent. Please move the point, or setup a "lead" to exit the QComponent area. 2. none of the 4 routing possibilities of this algorithm (^|_, ^^|, __|, _|^) can complete. Please use Pathfinder instead


QiskitMetalDesignError: Qiskit Metal - Designer Error: User action required. 
connect_simple() has failed. This might be due to one of two reasons. 1. Either one of the start point [-0.8 -0.4] or the end point [-0.115 -0.41 ] provided are inside the bounding box of another QComponent. Please move the point, or setup a "lead" to exit the QComponent area. 2. none of the 4 routing possibilities of this algorithm (^|_, ^^|, __|, _|^) can complete. Please use Pathfinder instead